In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 05:50:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark.version

'4.1.1'

In [2]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-09 05:50:51--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.170.186.111, 3.170.186.198, 3.170.186.229, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.170.186.111|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M   196MB/s    in 0.3s    

2026-03-09 05:50:52 (196 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [9]:
spark.read.parquet("yellow_tripdata_2025-11.parquet").repartition(4) \
        .write.parquet('data/EDA', mode='overwrite')

In [10]:
df_yellow = spark.read.option("recursiveFileLookup", "true").parquet("data/EDA/")

In [11]:
df_yellow.registerTempTable('yellow')

/home/abrar/spark-batch-job/.venv/lib/python3.12/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [12]:
df_yellow.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-07 18:37:45|  2025-11-07 18:41:51|              1|         0.78|         1|                 N|         140|    

In [13]:
df_2 = spark.sql("""
SELECT 
    COUNT(*) NUM_OF_TRIPS
FROM
    yellow
WHERE
    tpep_pickup_datetime >= '2025-11-15 00:00:00' AND tpep_pickup_datetime < '2025-11-16 00:00:00'
""")

In [15]:
df_2.show()

+------------+
|NUM_OF_TRIPS|
+------------+
|      162604|
+------------+



In [16]:
df_yellow.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [17]:
df_yellow

DataFrame[VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double, cbd_congestion_fee: double]

In [18]:
from pyspark.sql import functions as F

# 'end_time' is the timestamp from which 'start_time' is subtracted
df_3 = df_yellow.withColumn(
    "hours_diff",
    F.timestamp_diff("HOUR", F.col("tpep_pickup_datetime"), F.col("tpep_dropoff_datetime"))
)


In [20]:
df_3.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee',
 'hours_diff']

In [25]:
df_3.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|hours_diff|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------+
|       2| 2025-11-07 18:37:45|  2025-11-07 18:41:51|              1|         0.78|         1|   

In [21]:
df_3.registerTempTable('hours_elapsed')

In [26]:
df_4 = spark.sql("""
SELECT 
    MAX(hours_diff)
FROM
    hours_elapsed
""")

In [27]:
df_4.show()

[Stage 21:>                                                         (0 + 4) / 4]

+---------------+
|max(hours_diff)|
+---------------+
|             90|
+---------------+



In [28]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-09 06:29:07--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.170.186.229, 3.170.186.198, 3.170.186.41, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.170.186.229|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-09 06:29:07 (109 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [35]:
df_taxi_zone = spark.read \
    .option("header", "true") \
    .option("inferSchema", True) \
    .csv('taxi_zone_lookup.csv')

In [36]:
df_taxi_zone

DataFrame[LocationID: int, Borough: string, Zone: string, service_zone: string]

In [38]:
df_result = df_taxi_zone.join(df_yellow, df_taxi_zone.LocationID == df_yellow.PULocationID)

In [39]:
df_yellow.count()

4181444

In [40]:
df_result.count()

4181444

In [41]:
df_result

DataFrame[LocationID: int, Borough: string, Zone: string, service_zone: string, VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double, cbd_congestion_fee: double]

In [42]:
df_result.registerTempTable('final_table')

In [47]:
spark.sql("""
SELECT 
 COUNT(*) AS COUNT,
 Zone 
FROM
    final_table
GROUP BY Zone 
ORDER BY COUNT 
""").show()

+-----+--------------------+
|COUNT|                Zone|
+-----+--------------------+
|    1|       Arden Heights|
|    1|Governor's Island...|
|    1|Eltingville/Annad...|
|    3|       Port Richmond|
|    4|         Great Kills|
|    4| Green-Wood Cemetery|
|    4|   Rossville/Woodrow|
|    4|       Rikers Island|
|    5|         Jamaica Bay|
|   12|         Westerleigh|
|   14|       West Brighton|
|   14|New Dorp/Midland ...|
|   14|             Oakwood|
|   14|        Crotona Park|
|   15|       Willets Point|
|   16|Breezy Point/Fort...|
|   17|Saint George/New ...|
|   18|       Broad Channel|
|   21|     Mariners Harbor|
|   22|Heartland Village...|
+-----+--------------------+
only showing top 20 rows
